In [12]:
import torch
import torch.nn as nn

In [13]:
class GELU(nn.Module):
    ''' 
    A variant of relu that allows some range of negative numbers
    '''
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return 0.5 * x * (
            1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3)))
        )

In [14]:
class DeepNeuralNetwork(nn.Module):
    '''
    Residual connections that add accense of the original input embedding on every layer
    '''
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, X):
        original_input = X

        for layer in self.layers:
            layer_output = layer(X)

            if self.use_shortcut and original_input.shape == layer_output.shape:
                X = original_input + layer_output
            else:
                X = layer_output

        return X

In [ ]:
# experimented with
def forward(self, X):
    original_input = X

    for layer in self.layers:
        layer_output = layer(X)

        if self.use_shortcut and original_input.shape == layer_output.shape:
            X = original_input + layer_output
        else:
            X = layer_output

    return X

In [ ]:
# Original forward pass 
def forward(self, X):
    for layer in self.layers:
        layer_output = layer(X)
        if self.use_shortcut and X.shape == layer_output.shape:
            X = X + layer_output
        else:
            X = layer_output
    return X

In [15]:
layer_sizes = [3, 3, 3, 3, 3, 1]
sample_input = torch.tensor([[1., 0., -1.]])
torch.manual_seed(123)
model_without_shortcut = DeepNeuralNetwork(layer_sizes, use_shortcut=False)

In [16]:
def print_gradients(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [17]:
print_gradients(model_without_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.00020173587836325169
layers.1.0.weight has gradient mean of 0.00012011159560643137
layers.2.0.weight has gradient mean of 0.0007152040489017963
layers.3.0.weight has gradient mean of 0.0013988736318424344
layers.4.0.weight has gradient mean of 0.005049645435065031


In [18]:
torch.manual_seed(123)
model_with_shortcut = DeepNeuralNetwork(
layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)

layers.0.0.weight has gradient mean of 0.0011413325555622578
layers.1.0.weight has gradient mean of 0.004565919749438763
layers.2.0.weight has gradient mean of 0.026039404794573784
layers.3.0.weight has gradient mean of 0.12043901532888412
layers.4.0.weight has gradient mean of 0.28059855103492737


In [25]:
def print_gradients2(model, x):
    output = model(x)
    target = torch.tensor([[0.]])
    loss = nn.MSELoss()
    loss = loss(output, target)
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(f"{name} has gradient norm of {param.grad.norm().item()}")

In [26]:
torch.manual_seed(123)
model_without_shortcut2 = DeepNeuralNetwork(layer_sizes, False)

torch.manual_seed(123)
model_with_shortcut2 = DeepNeuralNetwork(layer_sizes, True)

In [27]:
print("Without shortcut: ")
print_gradients2(model_without_shortcut2, sample_input)

print("With shortcut: ")
print_gradients2(model_with_shortcut2, sample_input)

Without shortcut: 
layers.0.0.weight has gradient norm of 0.0008081949199549854
layers.1.0.weight has gradient norm of 0.0004058456397615373
layers.2.0.weight has gradient norm of 0.0031206475105136633
layers.3.0.weight has gradient norm of 0.00520906038582325
layers.4.0.weight has gradient norm of 0.00981670431792736
With shortcut: 
layers.0.0.weight has gradient norm of 0.005356376059353352
layers.1.0.weight has gradient norm of 0.02274719998240471
layers.2.0.weight has gradient norm of 0.09393739700317383
layers.3.0.weight has gradient norm of 0.37322497367858887
layers.4.0.weight has gradient norm of 0.6003844738006592
